In [ ]:
# import micropip
# await micropip.install("matplotlib")
# await micropip.install("scipy")
# await micropip.install("ipywidgets")

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

%matplotlib inline

def correlation(f, g, ih):
    f = f.astype(np.complex128)
    g = g.astype(np.complex128)
    g_shifted = np.roll(g, ih)
    corr = np.sum(np.conjugate(g_shifted) * f)
    return corr

def full_correlation(f, g):
    N = f.shape[0]
    full_corr = np.zeros(N, dtype=np.complex128)
    for ih in range(-N // 2, N // 2):
        full_corr[ih + N // 2] = correlation(f, g, ih)
    return full_corr


def representacion(ih_time):
    """ih_time: shift in the same units as the global time array t.
       Uses global dt to convert to integer index ih."""
    ih = int(round(ih_time / dt))
    N = f.shape[0]
    g_shifted = np.roll(g, ih)
    integrand = np.conj(g_shifted) * f

    fig, ax = plt.subplots(2, 2, figsize=(7, 5))

    # Top-left: f
    ax[0, 0].plot(t, np.real(f), label='Re(f)')
    ax[0, 0].plot(t, np.abs(f), '--', label='|f|')
    ax[0, 0].set_title('f(t)')
    ax[0, 0].legend(fontsize=8)
    #ax[0, 0].set_xlim(0, N)

    # Top-right: g and shifted g
    ax[0, 1].plot(t, np.real(g), label='Re(g)', alpha=0.4)
    ax[0, 1].plot(t, np.real(g_shifted), label=f'Re(g shifted, h={ih_time:.2f})')
    ax[0, 1].plot(t, np.abs(g_shifted), '--', label='|g shifted|')
    ax[0, 1].set_title(f'g(t) shifted by h = {ih_time:.2f}')
    ax[0, 1].legend(fontsize=8)

    # Bottom-left: integrand
    ax[1, 0].plot(t, np.abs(g_shifted), alpha=0.4, label='|g shifted|')
    ax[1, 0].plot(t, np.abs(f), alpha=0.4, label='|f|')
    ax[1, 0].plot(t, np.real(integrand), label='Re(g* · f)', linewidth=2)
    #ax[1, 0].plot(t, np.abs(integrand), '--', label='|g* · f|', linewidth=2)
    ax[1, 0].fill_between(t, np.real(integrand), alpha=0.2)
    ax[1, 0].set_title(f'Integrand  →  C(h={ih_time:.2f}) = {correlation(f, g, ih):.3f}')
    ax[1, 0].legend(fontsize=8)

    # Bottom-right: full correlation — x-axis in time units
    ih_axis_time = ih_axis * dt
    ax[1, 1].plot(ih_axis_time, np.abs(full_corr), label=f'|C(h)|={np.abs(full_corr[int(ih_time/dt+N/2)]):.2f}')
    #ax[1, 1].plot(ih_axis_time, np.real(full_corr), label='Re(C(h))', alpha=0.6)
    ax[1, 1].axvline(x=ih_time, color='r', linestyle='--', label=f'h={ih_time:.2f}')
    ax[1, 1].scatter([ih_time], [np.abs(full_corr[ih + N // 2])], c='r', zorder=5)
    ax[1, 1].set_title('Correlation C(h)')
    ax[1, 1].legend(fontsize=8)
    ax[1, 1].set_xlabel('h (time units)')

    plt.tight_layout()
    plt.show()


npts = 1000
t = np.linspace(-50, 50, npts)
dt = t[1] - t[0]               # time step (fs) — used by representacion
ix = np.arange(npts)

ih_axis = np.arange(-npts // 2, npts // 2)

slider = widgets.FloatSlider(
    value=0.0, min=t[0], max=t[-1], step=dt,
    description='h (fs):', continuous_update=True,
    layout=widgets.Layout(width='450px')
)

btn_left  = widgets.Button(description='◀', layout=widgets.Layout(width='40px'))
btn_right = widgets.Button(description='▶', layout=widgets.Layout(width='40px'))

def step_left(b):
    slider.value = max(slider.min, slider.value - slider.step)

def step_right(b):
    slider.value = min(slider.max, slider.value + slider.step)

btn_left.on_click(step_left)
btn_right.on_click(step_right)

# El producto de correlación

Elm producto de correlación es una medida de la regularidad relativa entre dos funciones. Es por eso que también se le considera una forma de avaluar la coherencia entre dos funciones.

Nosotros lo aplicaremos a campos electromagnéticos en el tiempo.

Tomamos un eje temporal que se extiende durante 100 $fs$ (un $fs$ es $10^{-15}$ s). Típicamente 

## Correlación de dos envolventes

Comencemos evaluando la correlación entre dos funciones arbitrarias, que podemos considerar envolventes gaussianas de dos ondas, con anchuras $\sigma_f$ y $\sigma_g$

$f= e^{-(t-t_0)^2/2 \sigma_f^2}$

$g= e^{-(t-t_0)^2/2 \sigma_g^2}$

Observa que la semianchura a media altura viene dada por 

$f=1/2 \rightarrow (t-t_0)^2/2 \sigma_f^2 = - \log 1/2 \rightarrow  t-t_0 = \sigma_f \sqrt(2 \log 2) = 0.78\sigma_f$

por lo que la achura a media altura viene dada por $1.56 \sigma_f$

1. En el caso en el que $\sigma_g=\sigma_f=\sigma$, determina la anchura a media altura del producto de correlación y relacionalo con la anchura del pulso. NOTA: comienza con $\sigma=5$ por ejemplo.



In [33]:

sigmaf=5
f = np.exp(-t**2/2/sigmaf**2)
sigmag=sigmaf
g = np.exp(-t**2/2/sigmag**2)
full_corr = full_correlation(f, g)

out = widgets.interactive_output(representacion, {'ih_time': slider})
controls = widgets.HBox([btn_left, slider, btn_right])
display(widgets.VBox([controls, out]))

plt.close()

## Correlación de dos ondas

Si en lugar de las envolventes hacemos la correlación de dos ondas, de frecuencias $\omega_f$ y $\omega_g$ e igual anchura $\sigma=5$

$f= e^{-(t-t_0)^2/2/\sigma} e^{-i \omega_f t}$

$g= e^{-(t-t_0)^2/2/\sigma} e^{-i \omega_g t}$

tendremos la correlación de ambas ondas. 

1- Prueba a hacer la correlación con frecuencias iguales ($\omega_f=\omega_g=0.2 fs^{-1}$)

2- ¿qué ocurre en el caso en el que $\omega_g=2\omega_f$?

In [34]:

sigmaf=5
wf=2
f = np.exp(-t**2/2/sigmaf**2)*np.exp(1j*wf*t)
sigmag=sigmaf
wg=wf*2
g = np.exp(-t**2/2/sigmag**2)*np.exp(1j*wg*t)
full_corr = full_correlation(f, g)

out = widgets.interactive_output(representacion, {'ih_time': slider})
controls = widgets.HBox([btn_left, slider, btn_right])
display(widgets.VBox([controls, out]))

plt.close()

## Autocorrelación de una onda real.

Hemos visto que podemos calcular la autocorrelación en el laboratorio usando un cristal nolineal.

Calcula la autocorrelación de un campo expresado ahora como real, no como una amplitud compleja



In [35]:

sigmaf=5
wf=2
f = np.exp(-t**2/2/sigmaf**2)*np.cos(wf*t)
sigmag=sigmaf
wg=wf
g = np.exp(-t**2/2/sigmag**2)*np.cos(wg*t)
full_corr = full_correlation(f, g)

out = widgets.interactive_output(representacion, {'ih_time': slider})
controls = widgets.HBox([btn_left, slider, btn_right])
display(widgets.VBox([controls, out]))

plt.close()

## Coherencia temporal

El tiempo en el que decae la autocorrelación de un campo a la mita se denomina __tiempo de coherencia__. Hemos visto que el el caso de un pulso de frecuancia fija, el tiempo de coherencia es aproximadamente la duración del pulso. 

__El tiempo de coherencia nos da el retardo máximo que tienen que tener dos pulsos para que forman interferencias__

Vamos a volver al caso de amplitudes complejas, ya que la función de coherencia es más suave.

Calcula la autocorrelación de un campo que contiene una fase fruto de una pequeña desviación aleatoria $d\varphi$. Obtén el tiempo de correlación para diferentes $d \varphi$

In [42]:

sigmaf=5
wf=2

dphi=0.1*np.pi
phi=dphi*np.cumsum(np.random.uniform(0, 1, size=npts))
f = np.exp(-t**2/2/sigmaf**2)*np.exp(-1j*wf*t)*np.exp(-1j*phi)
sigmag=sigmaf
wg=wf
g = np.exp(-t**2/2/sigmag**2)*np.exp(-1j*wf*t)*np.exp(-1j*phi)
full_corr = full_correlation(f, g)


out = widgets.interactive_output(representacion, {'ih_time': slider})
controls = widgets.HBox([btn_left, slider, btn_right])
display(widgets.VBox([controls, out]))

plt.close()